# COJO conditional analyses at *APOE* and *SNCA*
- **Project:** Genetic determinants of Lewy body pathology
- **Version:** Python/3.10
- **Last Updated:** 16-SEP-2026

## Notebook Overview
Conditional and joint analyses (GCTA-COJO) of the Lewy body disease (LBD) case-control GWAS meta-analysis
at two loci, using unrelated **GP2 release 11 EUR controls** (Tier 2) as the LD reference (hg38).

- **Part 1:** load the GWAS summary statistics and build the GP2 unrelated-control keep list
- **Part 2 (*APOE*):** independent signals, and conditioning on the ε4 SNP rs429358
- **Part 3 (*SNCA*):** is the LBD lead rs2737030 independent of the PD lead rs356182 and of the RBD / DLB leads?
- **Part 4 (*SNCA*):** input tables for the regional association figures (unconditional and conditioned on rs2737030)


**Conventions**
- GWAS and GP2 are both hg38; variants are matched on position + allele set, and rsIDs are labels only.
- Effect allele = `Allele1`; `b`/`se` are log odds. Case-control COJO uses effective N = 4·N<sub>case</sub>·N<sub>ctrl</sub> / (N<sub>case</sub> + N<sub>ctrl</sub>).
- The GP2 reference overlaps the GWAS sample; this is appropriate for COJO and is reported in the Methods.

**Requirements:** `plink2` (or PLINK 1.9) and `gcta64` on `PATH`; Python with pandas, numpy and matplotlib.

In [ ]:
import os, re, glob, shutil, subprocess
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# ── Config — edit paths for your workspace ───────────────────────────────────
PLINK = shutil.which("plink2") or shutil.which("plink")    
assert PLINK, "No plink/plink2 binary found"
GCTA = "gcta64"
assert shutil.which(GCTA), f"{GCTA} not on PATH"

# GP2 release 11 (hg38)
GP2_DIR     = "PATH/TO/gp2_tier2_eu_release11"               # GP2 release 11 (Tier 2)
GP2_PFILE   = os.path.join(GP2_DIR, "imputed_genotypes/EUR/chr{chrom}_EUR_release11_vwb")  # .pgen/.pvar/.psam prefix
GP2_CLIN    = os.path.join(GP2_DIR, "clinical_data/master_key_release11_final_vwb.csv")
GP2_RELATED = os.path.join(GP2_DIR, "meta_data/related_samples/EUR_release11_vwb.related")

# LBD case-control GWAS (hg38)
META_DIR       = "PATH/TO/meta-analysis"
GWAS_FILE      = os.path.join(META_DIR, "LBD_casecontrol_meta.filtered.tab")  # COJO input: CHR BP Allele1 Allele2 Freq1 Effect StdErr P-value
GWAS_PLOT_FILE = os.path.join(META_DIR, "META_casecontrol1.tbl")              # METAL output, Part 4 regional plots
N_CASE, N_CTRL = 2658, 12068

# Outputs
OUT_DIR     = os.path.join(META_DIR, "COJO")
GP2_REF_DIR = os.path.join(OUT_DIR, "GP2_REF")
PLOT_DIR    = os.path.join(META_DIR, "plots")
for d in (OUT_DIR, GP2_REF_DIR, PLOT_DIR):
    os.makedirs(d, exist_ok=True)

# Analysis parameters
N_EFF       = int(4 * N_CASE * N_CTRL / (N_CASE + N_CTRL))
P_THRESHOLD = 5e-8
COLLINEAR   = 0.9                          # --cojo-collinear
DIFF_FREQ   = 0.2                          # --diff-freq
REF_QC      = ["--geno", 0.05, "--maf", 0.01]

print("plink:", PLINK)
print(f"N_eff = {N_EFF:,}")

In [ ]:
# ── Helpers ──────────────────────────────────────────────────────────────────
def run(cmd, quiet=False, check=True):
    """Run a command. GCTA logs its errors to stdout, so both streams are shown on failure."""
    cmd = list(map(str, cmd))
    if not quiet:
        print("\n+", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    if not quiet:
        print(r.stdout[-1500:])
    if r.returncode:
        print("+ FAILED:", " ".join(cmd))
        print("---- stdout ----\n" + r.stdout[-2000:])
        print("---- stderr ----\n" + r.stderr[-1000:])
        if check:
            raise RuntimeError(" ".join(cmd))
    return r

def write_ids(path, ids):
    with open(path, "w") as fh:
        fh.write("\n".join(ids) + "\n")
    return path

def read_bim(bfile):
    bim = pd.read_csv(bfile + ".bim", sep=r"\s+", header=None,
                      names=["chr", "id", "cm", "bp", "a1", "a2"])
    bim["a1"], bim["a2"] = bim["a1"].str.upper(), bim["a2"].str.upper()
    bim["aset"] = [frozenset(x) for x in zip(bim["a1"], bim["a2"])]
    return bim

def n_samples(bfile):
    with open(bfile + ".fam") as fh:
        return sum(1 for _ in fh)

def gp2_id(bim, bp, alleles=None, label=""):
    """Resolve a position (+ optional allele set) to its GP2 variant ID."""
    hit = bim[bim["bp"] == bp]
    assert not hit.empty, (f"{label} {bp:,} not in the GP2 region bfile — "
                           f"check the build (hg38) or widen the region")
    if alleles:
        h = hit[hit["aset"] == frozenset(alleles)]
        assert not h.empty, (f"{label} {bp:,} present but alleles are "
                             f"{hit[['a1', 'a2']].values.tolist()}, not {alleles}")
        hit = h
    if len(hit) > 1:
        print(f"  ! {len(hit)} variants at {bp:,}: {hit['id'].tolist()} — using the first")
    return hit["id"].iloc[0]

def rel_to(bp, ref_bp):
    """Distance and orientation vs ref_bp for a minus-strand gene (larger coordinate = 5′)."""
    d = bp - ref_bp
    return (0, "-") if d == 0 else (abs(d), "5′" if d > 0 else "3′")

_SUP = str.maketrans("0123456789-", "⁰¹²³⁴⁵⁶⁷⁸⁹⁻")

def fmt_p(p):
    """Format p-value for display in scientific notation"""
    p = float(p)
    if not np.isfinite(p) or p <= 0:
        return "< 1 × 10⁻³⁰⁰"
    e = int(np.floor(np.log10(p))); m = p / 10.0 ** e
    if m >= 9.95:
        m, e = 1.0, e + 1
    return f"{m:.1f} × 10{str(e).translate(_SUP)}"

# plink2 prints "r^2 = …  |D'| = …"; PLINK 1.9 prints "R-sq = …  D' = …"
_R2_RE = re.compile(r"(?:R-sq|r\^2)\s*=\s*([\d.eE+-]+)")
_DP_RE = re.compile(r"\|?D'\|?\s*=\s*([\d.eE+-]+)")

def pairwise_ld(bfile, id1, id2):
    """(r², D′) between two variants in the reference panel."""
    r = subprocess.run([PLINK, "--bfile", bfile, "--ld", id1, id2,
                        "--out", os.path.join(OUT_DIR, "_ld_tmp")],
                       capture_output=True, text=True)
    txt = r.stdout + r.stderr
    m1, m2 = _R2_RE.search(txt), _DP_RE.search(txt)
    assert m1 and m2, f"could not parse --ld output for {id1} vs {id2}:\n{txt[-800:]}"
    return float(m1.group(1)), float(m2.group(1))

In [ ]:
# ── Reference panel, .ma and COJO wrappers ───────────────────────────────────
def build_region_ref(chrom, region, locus):
    """GP2 unrelated-control bfile for one chromosome, then restricted to the locus region."""
    chr_bfile    = os.path.join(GP2_REF_DIR, f"chr{chrom}_GP2_REF")
    region_bfile = os.path.join(GP2_REF_DIR, f"GP2_REF_{locus}")
    if not os.path.exists(chr_bfile + ".bed"):
        run([PLINK, "--pfile", GP2_PFILE.format(chrom=chrom), "--keep", REF_ID_PATH,
             *REF_QC, "--threads", 8, "--memory", 26000,
             "--make-bed", "--out", chr_bfile])
    if not os.path.exists(region_bfile + ".bed"):
        run([PLINK, "--bfile", chr_bfile, "--chr", chrom,
             "--from-bp", region[0], "--to-bp", region[1],
             "--make-bed", "--out", region_bfile])
    print(f"{locus} reference: {region_bfile}  ({n_samples(region_bfile):,} samples)")
    return chr_bfile, region_bfile

def make_ma(bim, chrom, region, path, min_match=0.5):
    """Join GWAS → GP2 on position + allele set and write a COJO .ma (A1 = Allele1).
    GCTA resolves A1/A2 orientation against the bfile."""
    g = gwas[(gwas["CHR"] == chrom) & gwas["BP"].between(*region)].copy()
    g["aset"] = [frozenset(x) for x in zip(g["Allele1"], g["Allele2"])]
    m = g.merge(bim[["bp", "aset", "id"]], left_on=["BP", "aset"], right_on=["bp", "aset"])
    print(f"GP2 region variants: {len(bim):,} | GWAS variants in region: {len(g):,} | "
          f"matched on pos+alleles: {len(m):,} ({len(m) / len(bim):.0%} of GP2)")
    assert len(m) > min_match * len(bim), "Low match rate — check build/allele coding"
    ma = pd.DataFrame({"SNP": m["id"], "A1": m["Allele1"], "A2": m["Allele2"],
                       "freq": m["Freq1"], "b": m["Effect"], "se": m["StdErr"],
                       "p": m["P-value"], "N": N_EFF}).dropna().drop_duplicates("SNP")
    ma.to_csv(path, sep=" ", index=False)
    print(f"{len(ma):,} SNPs → {path}")
    return ma

def cojo(bfile, ma, out, *args, collinear=COLLINEAR, quiet=False):
    return run([GCTA, "--bfile", bfile, "--cojo-file", ma, *args,
                "--cojo-collinear", collinear, "--diff-freq", DIFF_FREQ, "--out", out],
               quiet=quiet, check=False)

def cojo_slct(bfile, ma, out):
    """Stepwise selection of independent signals → .jma.cojo table."""
    r = cojo(bfile, ma, out, "--cojo-slct", "--cojo-p", P_THRESHOLD)
    assert r.returncode == 0 and os.path.exists(out + ".jma.cojo"), "--cojo-slct failed"
    return pd.read_csv(out + ".jma.cojo", sep="\t")

def cojo_cond(bfile, ma, cond_ids, out, target_id=None, collinear=COLLINEAR, quiet=True):
    """Condition every variant on cond_ids.
    Returns (cma table, status), or (target row, status) when target_id is given.
    status ∈ ok | target_dropped | gcta_error: …   (a --cojo-cond SNP absent from the .ma is silently ignored)"""
    cf = write_ids(out + ".cond.txt", cond_ids)
    r = cojo(bfile, ma, out, "--cojo-cond", cf, collinear=collinear, quiet=quiet)
    if r.returncode or not os.path.exists(out + ".cma.cojo"):
        lines = [l for l in (r.stdout + "\n" + r.stderr).splitlines() if l.strip()]
        return None, "gcta_error: " + (lines[-1][:200] if lines else f"exit {r.returncode}")
    cma = pd.read_csv(out + ".cma.cojo", sep="\t")
    if target_id is None:
        return cma, "ok"
    row = cma[cma["SNP"] == target_id]
    return (None, "target_dropped") if row.empty else (row.iloc[0], "ok")

def cojo_joint(bfile, ma, ids, out):
    """Joint model of exactly the given SNPs → .jma.cojo table, or None if GCTA produced none."""
    ef = write_ids(out + ".snps.txt", ids)
    r = cojo(bfile, ma, out, "--extract", ef, "--cojo-joint", quiet=True)
    if r.returncode or not os.path.exists(out + ".jma.cojo"):
        return None
    return pd.read_csv(out + ".jma.cojo", sep="\t")

## Part 1 — GWAS summary statistics and GP2 control keep-list
Unrelated GP2 samples with baseline phenotype *Control* or *Population Control*. The keep-list is
shared by every chromosome-level reference built below.

In [ ]:
gwas = pd.read_csv(GWAS_FILE, sep="\t", low_memory=False)
gwas.columns = gwas.columns.str.strip()
gwas = gwas.dropna(subset=["CHR", "BP", "Allele1", "Allele2", "Freq1", "Effect", "StdErr", "P-value"])
gwas["CHR"] = gwas["CHR"].astype(float).astype(int)
gwas["BP"]  = gwas["BP"].astype(int)
for c in ("Allele1", "Allele2"):
    gwas[c] = gwas[c].str.upper()
for c in ("Freq1", "Effect", "StdErr", "P-value"):
    gwas[c] = pd.to_numeric(gwas[c], errors="coerce")
print(f"GWAS variants: {len(gwas):,}")

clin = pd.read_csv(GP2_CLIN, low_memory=False)
controls = clin[clin["baseline_GP2_phenotype"].isin(["Control", "Population Control"])]
controls = controls.rename(columns={"FID": "#FID", "GP2ID": "IID"})
related  = pd.read_csv(GP2_RELATED)
controls = controls[~controls["IID"].isin(related["IID1"])].drop_duplicates(subset="IID")

REF_ID_PATH = os.path.join(GP2_REF_DIR, "REF_ID.txt")
controls[["#FID", "IID"]].to_csv(REF_ID_PATH, sep=" ", index=False, header=False)
print(f"Unrelated controls in keep-list: {len(controls):,}")

## Part 2 — *APOE*

In [ ]:
APOE_CHR     = 19
APOE_REGION  = (44_830_000, 44_930_000)
E4_RS, E4_BP = "rs429358", 44_908_684      # conditioning variant
E2_RS, E2_BP = "rs7412",   44_908_822      # diagnostic

def annotate_apoe(bp):
    # approximate hg38 gene bounds
    if bp <  44_889_226: return "NECTIN2"
    if bp <= 44_903_689: return "TOMM40"
    if bp <= 44_909_393: return "APOE"
    if bp <= 44_919_349: return "APOC1"
    return "APOC1P1/downstream"

### 2a. LD reference and `.ma`

In [ ]:
APOE_CHR_BFILE, APOE_BFILE = build_region_ref(APOE_CHR, APOE_REGION, "APOE")
apoe_bim = read_bim(APOE_BFILE)

APOE_MA = os.path.join(OUT_DIR, "APOE_GP2_LBD.ma")
apoe_ma = make_ma(apoe_bim, APOE_CHR, APOE_REGION, APOE_MA)

### 2b. Independent signals (`--cojo-slct`)

In [ ]:
apoe_jma = cojo_slct(APOE_BFILE, APOE_MA, os.path.join(OUT_DIR, "APOE_GP2_slct"))
apoe_jma["gene"] = apoe_jma["bp"].map(annotate_apoe)

print(f"\nIndependent signals: {len(apoe_jma)}")
print(apoe_jma[["SNP", "bp", "gene", "freq", "b", "se", "p", "bJ", "bJ_se", "pJ"]].to_string(index=False))
apoe_jma.to_csv(os.path.join(OUT_DIR, "APOE_GP2_independent_signals.tsv"), sep="\t", index=False)

### 2c. Conditioning on rs429358 (ε4)
`pC` is the conditional P. Any variant still below 5×10⁻⁸ is a signal independent of ε4.

In [ ]:
e4_id = gp2_id(apoe_bim, E4_BP, label=E4_RS)
assert e4_id in set(apoe_ma["SNP"]), f"{E4_RS} ({e4_id}) not in the .ma"
print(f"{E4_RS} → {e4_id}  (marginal P = {fmt_p(apoe_ma.set_index('SNP').loc[e4_id, 'p'])})")

apoe_cma, status = cojo_cond(APOE_BFILE, APOE_MA, [e4_id],
                             os.path.join(OUT_DIR, f"APOE_cond_{E4_RS}"), quiet=False)
assert status == "ok", status
apoe_cma["gene"] = apoe_cma["bp"].map(annotate_apoe)

cols = ["SNP", "bp", "gene", "freq", "b", "p", "bC", "bC_se", "pC"]
survivors = apoe_cma[apoe_cma["pC"] < P_THRESHOLD].sort_values("pC")
print(f"\nVariants genome-wide significant after conditioning on {E4_RS}: {len(survivors)}")
print(f"\nTop 10 by conditional P:")
print(apoe_cma.nsmallest(10, "pC")[cols].to_string(index=False))

# ε2 diagnostic
print(f"\n{E2_RS} (chr{APOE_CHR}:{E2_BP:,}):")
e2 = apoe_bim[apoe_bim["bp"] == E2_BP]
if e2.empty:
    print("  not in the GP2 reference panel")
else:
    e2_id = e2["id"].iloc[0]
    in_ma, in_cma = apoe_ma[apoe_ma["SNP"] == e2_id], apoe_cma[apoe_cma["SNP"] == e2_id]
    print("  GP2 id:", e2_id)
    print(f"  marginal P = {fmt_p(in_ma['p'].iloc[0])}" if len(in_ma)
          else "  absent from the .ma (not in the GWAS, or allele mismatch)")
    if len(in_cma):
        print(f"  conditional P on {E4_RS} = {fmt_p(in_cma['pC'].iloc[0])}")

### 2d. Marginal vs conditional association

In [ ]:
d = apoe_cma.dropna(subset=["p", "pC"])
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
for ax, col, title in zip(axes, ["p", "pC"], ["Marginal", f"Conditional on {E4_RS}"]):
    ax.scatter(d["bp"] / 1e6, -np.log10(d[col].clip(lower=1e-300)), s=10, alpha=0.7, color="steelblue")
    ax.axhline(-np.log10(P_THRESHOLD), color="red", ls="--", lw=0.8)
    ax.axvline(E4_BP / 1e6, color="darkorange", ls=":", lw=1.2, label=E4_RS)
    ax.set_ylabel("-log10(P)")
    ax.set_title(f"APOE locus — {title}")
    ax.legend(fontsize=8)
axes[-1].set_xlabel(f"chr{APOE_CHR} position (Mb, hg38)")
plt.tight_layout()
fig.savefig(os.path.join(PLOT_DIR, "APOE_GP2_cond_manhattan.png"), dpi=150, bbox_inches="tight")
plt.show()

## Part 3 — SNCA

Tests whether the LBD SNCA lead **rs2737030** (SNCA intron 4) is a distinct signal from the canonical
PD lead and from the RBD and DLB SNCA leads.

| variant | position (hg38) | role |
|---|---|---|
| rs2737030 | chr4:89,788,948 | **target** — LBD SNCA lead |
| rs356182 | chr4:89,704,960 | canonical PD lead, 3′ of SNCA |
| rs3756059 | chr4:89,836,121 | RBD lead, SNCA gene body |
| rs7680557 | chr4:89,842,209 | DLB lead, 5′ of SNCA |

In [ ]:
SNCA_CHR    = 4
SNCA_REGION = (89_650_000, 89_954_629)       # 3′ of rs356182 → SNCA → MMRN1

TARGET_RS = "rs2737030"
SNCA_VARIANTS = {                             # alleles=None → match on position only
    "rs2737030": dict(bp=89_788_948, alleles={"C", "T"}, role="LBD lead (target)"),
    "rs356182":  dict(bp=89_704_960, alleles={"G", "A"}, role="PD lead"),
    "rs3756059": dict(bp=89_836_121, alleles=None,       role="RBD lead"),
    "rs7680557": dict(bp=89_842_209, alleles=None,       role="DLB lead"),
}

# Summary statistics (from the unfiltered meta-analysis) for variants absent from GWAS_FILE.
# A --cojo-cond SNP with no .ma row is silently ignored, so every conditioning SNP must be present.
MA_OVERRIDES = {
    "rs356182": dict(A1="A", A2="G", freq=0.6541, b=-0.1122, se=0.0380, p=0.003201),
}

# Conditioning sets for the target. Pairs with r² > COLLINEAR are pruned to the SNP with the smaller P.
TARGET_SCENARIOS = [
    ["rs356182"],
    ["rs7680557"],
    ["rs3756059"],
    ["rs356182", "rs7680557"],
    ["rs356182", "rs7680557", "rs3756059"],
]
# Reciprocal runs: (target, conditioning set)
RECIPROCAL = [
    ("rs7680557", ["rs2737030"]),
    ("rs3756059", ["rs2737030"]),
    ("rs356182",  ["rs2737030"]),
    ("rs7680557", ["rs2737030", "rs356182"]),
    ("rs356182",  ["rs2737030", "rs7680557"]),
]
JOINT_PAIRS = [
    ("rs2737030", "rs356182"),
    ("rs2737030", "rs7680557"),
    ("rs7680557", "rs356182"),
]

def annotate_snca(bp):
    if bp <  89_724_099: return "3′ of SNCA"
    if bp <= 89_838_315: return "SNCA gene body"
    if bp <  89_879_532: return "SNCA 5′ / intergenic"
    if bp <= 89_954_629: return "MMRN1"
    return "other"

### 3a. LD reference and `.ma`

In [ ]:
SNCA_CHR_BFILE, SNCA_BFILE = build_region_ref(SNCA_CHR, SNCA_REGION, "SNCA")
snca_bim = read_bim(SNCA_BFILE)
N_REF    = n_samples(SNCA_BFILE)

SNCA_MA = os.path.join(OUT_DIR, "SNCA_GP2_LBD.ma")
snca_ma = make_ma(snca_bim, SNCA_CHR, SNCA_REGION, SNCA_MA)

### 3b. Resolve the variants in GP2

In [ ]:
V = {rs: dict(cfg, id=gp2_id(snca_bim, cfg["bp"], cfg["alleles"], rs))
     for rs, cfg in SNCA_VARIANTS.items()}
target_id = V[TARGET_RS]["id"]
rsmap     = {v["id"]: rs for rs, v in V.items()}

print(f"{'rsID':11s} {'GP2 id':22s} {'role':18s} {'location':22s} {'vs ' + TARGET_RS:>14s}")
for rs, v in V.items():
    d, side = rel_to(v["bp"], V[TARGET_RS]["bp"])
    where = "—" if d == 0 else f"{d / 1000:.1f} kb {side}"
    print(f"{rs:11s} {v['id']:22s} {v['role']:18s} {annotate_snca(v['bp']):22s} {where:>14s}")

### 3c. Make sure every variant is in the `.ma`

In [ ]:
present = set(snca_ma["SNP"])
rows, missing = [], []
for rs, v in V.items():
    if v["id"] in present:
        print(f"  {rs:11s} in .ma")
    elif rs in MA_OVERRIDES:
        rows.append(dict(SNP=v["id"], **MA_OVERRIDES[rs], N=N_EFF))
        print(f"  {rs:11s} added from MA_OVERRIDES")
    else:
        missing.append(rs)
        print(f"  {rs:11s} MISSING — add its summary statistics to MA_OVERRIDES")
assert TARGET_RS not in missing, "the target itself is missing from the .ma"

snca_ma_aug = pd.concat([snca_ma, pd.DataFrame(rows)], ignore_index=True) if rows else snca_ma
SNCA_MA_AUG = os.path.join(OUT_DIR, "SNCA_GP2_LBD_aug.ma")
snca_ma_aug.to_csv(SNCA_MA_AUG, sep=" ", index=False)
print(f"\n.ma → {SNCA_MA_AUG}  ({len(snca_ma_aug):,} SNPs)")

if missing:
    print(f"\n⚠ {missing} will be dropped from every conditioning set. Searching {META_DIR}:")
    want = [V[rs]["bp"] for rs in missing]
    for f in sorted(glob.glob(os.path.join(META_DIR, "*.tab")) + glob.glob(os.path.join(META_DIR, "*.tbl"))):
        try:
            if "BP" not in pd.read_csv(f, sep="\t", nrows=0).columns.str.strip():
                continue
            d = pd.read_csv(f, sep="\t", low_memory=False)
            d.columns = d.columns.str.strip()
            chrom = d["CHR"].astype(str).str.replace("chr", "").str.replace(r"\.0$", "", regex=True)
            hit = d[(chrom == str(SNCA_CHR)) & pd.to_numeric(d["BP"], errors="coerce").isin(want)]
            if not hit.empty:
                print("\n", os.path.basename(f))
                print(hit.to_string(index=False))
        except Exception as e:
            print(f"  (skipped {os.path.basename(f)}: {type(e).__name__})")

### 3d. Pairwise LD in GP2 EUR controls

In [ ]:
LD = {}
ld_tab = pd.DataFrame("—", index=list(V), columns=list(V), dtype=object)
for a, b in combinations(V, 2):
    r2, dp = pairwise_ld(SNCA_BFILE, V[a]["id"], V[b]["id"])
    LD[frozenset((a, b))] = (r2, dp)
    ld_tab.loc[a, b] = ld_tab.loc[b, a] = f"r²={r2:.3f} D′={dp:.2f}"

def ld_r2(a, b):
    return LD[frozenset((a, b))][0]

print(f"LD with the target {TARGET_RS} (GP2 EUR controls, n = {N_REF:,}):")
for rs in V:
    if rs != TARGET_RS:
        r2, dp = LD[frozenset((rs, TARGET_RS))]
        call = "same signal" if r2 > 0.8 else "partially shared" if r2 > 0.2 else "distinct"
        print(f"  {rs:11s} r² = {r2:.4f}  D′ = {dp:.3f}  → {call}")

display(ld_tab)
ld_tab.to_csv(os.path.join(OUT_DIR, "SNCA_LD_matrix.csv"))

### 3e. Independent signals (`--cojo-slct`)
The selected SNP need not be rs2737030 itself, so its LD with rs2737030 and rs356182 is also reported.

In [ ]:
snca_jma = cojo_slct(SNCA_BFILE, SNCA_MA_AUG, os.path.join(OUT_DIR, "SNCA_GP2_slct"))
snca_jma["rsID"]   = snca_jma["SNP"].map(rsmap).fillna("—")
snca_jma["region"] = snca_jma["bp"].map(annotate_snca)

print(f"\nIndependent signals: {len(snca_jma)}")
print(snca_jma[["rsID", "SNP", "bp", "region", "freq", "b", "se", "p", "bJ", "bJ_se", "pJ"]]
      .to_string(index=False))
for _, s in snca_jma.iterrows():
    for rs in (TARGET_RS, "rs356182"):
        if s["SNP"] != V[rs]["id"]:
            r2, dp = pairwise_ld(SNCA_BFILE, s["SNP"], V[rs]["id"])
            print(f"  {s['SNP']} vs {rs}: r² = {r2:.4f}  D′ = {dp:.3f}")

### 3f. Conditional analyses

In [ ]:
p_marg = snca_ma_aug.set_index("SNP")["p"]

def prune(rs_list):
    """Drop missing SNPs; of any pair with r² > COLLINEAR keep the one with the smaller marginal P."""
    kept = []
    for rs in sorted((x for x in rs_list if x not in missing),
                     key=lambda x: float(p_marg.get(V[x]["id"], 1.0))):
        clash = next((k for k in kept if ld_r2(rs, k) > COLLINEAR), None)
        if clash:
            print(f"    {rs} pruned — r² = {ld_r2(rs, clash):.3f} with {clash}")
        else:
            kept.append(rs)
    return kept

def condition(target_rs, cond_rs):
    ids  = [V[r]["id"] for r in cond_rs]
    stem = os.path.join(OUT_DIR, f"SNCA_cond_{target_rs}_on_{'_'.join(cond_rs)}")
    rec  = dict(target=target_rs, conditioned_on=" + ".join(cond_rs))
    note = f"--cojo-collinear {COLLINEAR}"
    row, status = cojo_cond(SNCA_BFILE, SNCA_MA_AUG, ids, f"{stem}_c{COLLINEAR}", V[target_rs]["id"])
    if status == "target_dropped":
        note = f"collinear at {COLLINEAR}; estimate from --cojo-collinear 0.99"
        row, status = cojo_cond(SNCA_BFILE, SNCA_MA_AUG, ids, f"{stem}_c0.99", V[target_rs]["id"],
                                collinear=0.99)
    if status.startswith("gcta_error"):
        return {**rec, "verdict": "failed", "note": status}
    if status == "target_dropped":
        return {**rec, "verdict": "same signal (collinear)", "note": "target collinear even at 0.99"}
    b, bC, pC = float(row["b"]), float(row["bC"]), float(row["pC"])
    return {**rec, "verdict": "survives" if pC < P_THRESHOLD else "does not survive",
            "b": b, "se": float(row["se"]), "p": float(row["p"]),
            "bC": bC, "bC_se": float(row["bC_se"]), "pC": pC,
            "attenuation_pct": round(100 * (1 - bC / b), 1), "note": note}

records, seen = [], set()
for tgt, conds in [(TARGET_RS, c) for c in TARGET_SCENARIOS] + RECIPROCAL:
    print(f"\n{tgt} | conditioned on {' + '.join(conds)}")
    kept = prune(conds)
    if not kept:
        print("  → no conditioning SNP available, skipped")
        continue
    if (tgt, frozenset(kept)) in seen:
        print(f"  → same as an earlier run ({' + '.join(kept)}), skipped")
        continue
    seen.add((tgt, frozenset(kept)))
    rec = condition(tgt, kept)
    records.append(rec)
    if "pC" in rec:
        print(f"  b {rec['b']:+.4f} → bC {rec['bC']:+.4f}   P {rec['p']:.2e} → pC {rec['pC']:.2e}   "
              f"{rec['verdict'].upper()}   [{rec['note']}]")
    else:
        print(f"  → {rec['verdict']}: {rec['note']}")

cond_summary = pd.DataFrame(records)
display(cond_summary)
cond_summary.to_csv(os.path.join(OUT_DIR, "SNCA_conditional_summary.csv"), index=False)

In [ ]:
# Results-text sentences for the target runs
for rec in records:
    if rec["target"] != TARGET_RS or "pC" not in rec:
        continue
    if rec["pC"] < P_THRESHOLD: verb = "remained genome-wide significant"
    elif rec["pC"] < 0.05:      verb = "was attenuated below genome-wide significance"
    else:                       verb = "was abolished"
    print(f"Conditioning on {rec['conditioned_on']}, the association at {TARGET_RS} {verb} "
          f"(conditional P = {fmt_p(rec['pC'])}, unconditional P = {fmt_p(rec['p'])}; "
          f"β {rec['b']:+.3f} → {rec['bC']:+.3f}, {rec['attenuation_pct']:.0f}% attenuation).\n")

### 3g. Joint models

In [ ]:
for pair in JOINT_PAIRS:
    tag = "_".join(pair)
    j = cojo_joint(SNCA_BFILE, SNCA_MA_AUG, [V[r]["id"] for r in pair if r not in missing],
                   os.path.join(OUT_DIR, f"SNCA_joint_{tag}"))
    if j is None:
        print(f"\n{' + '.join(pair)}: no joint output (collinear pair or GCTA error — see the .log)")
        continue
    j["rsID"], j["region"] = j["SNP"].map(rsmap).fillna("—"), j["bp"].map(annotate_snca)
    print(f"\n{' + '.join(pair)}:")
    print(j[["rsID", "bp", "region", "refA", "freq", "b", "se", "p", "bJ", "bJ_se", "pJ"]]
          .to_string(index=False))

## Part 4 — *SNCA* regional association figure inputs
Writes the tables used to draw the *SNCA* regional association plots (main figure and Supplementary Figure 2).

### 4a. Main-figure input

In [ ]:
PLOT_WINDOW_KB       = 200
REQUIRE_BOTH_COHORTS = False               # True → drop variants with "?" in METAL's Direction column
PLOT_TSV  = os.path.join(PLOT_DIR, "SNCA_regional_plot_input.tsv")

LEAD_BP   = V[TARGET_RS]["bp"]
HIGHLIGHT = {v["bp"]: rs for rs, v in V.items()}
lo, hi    = LEAD_BP - PLOT_WINDOW_KB * 1000, LEAD_BP + PLOT_WINDOW_KB * 1000

# 1. Resolve METAL column names
header = pd.read_csv(GWAS_PLOT_FILE, sep="\t", nrows=0).columns.tolist()

def pick(*cands):
    lut = {c.strip().lower(): c for c in header}
    return next((lut[c.lower()] for c in cands if c.lower() in lut), None)

COL = {"marker": pick("MarkerName", "SNP", "Variant", "ID"),
       "A1":     pick("Allele1", "A1", "EA", "effect_allele"),
       "A2":     pick("Allele2", "A2", "NEA", "other_allele"),
       "freq":   pick("Freq1", "EAF", "A1FREQ", "FreqA1"),
       "beta":   pick("Effect", "BETA", "b"),
       "se":     pick("StdErr", "SE"),
       "p":      pick("P-value", "P", "Pvalue", "P_value"),
       "chr":    pick("CHR", "Chr", "chromosome", "#CHROM"),
       "bp":     pick("BP", "POS", "position", "base_pair_location"),
       "dir":    pick("Direction")}
missing_cols = [k for k in ("A1", "A2", "beta", "se", "p") if not COL[k]]
assert not missing_cols, f"cannot find columns for {missing_cols} in {header}"
assert (COL["chr"] and COL["bp"]) or COL["marker"], f"no CHR/BP and no MarkerName to parse: {header}"

# 2. Chunked read of the plot window (positions from CHR/BP, else parsed from MarkerName)
MARKER_RE = r"^(?:chr)?([0-9]{1,2}|X|Y|MT|M)[:_\-]([0-9]+)"

def positions(df):
    if COL["chr"] and COL["bp"]:
        c, b = df[COL["chr"]], df[COL["bp"]]
    else:
        ex = df[COL["marker"]].astype(str).str.extract(MARKER_RE, flags=re.I)
        c, b = ex[0], ex[1]
    c = (c.astype(str).str.lower()
          .str.replace("^chr", "", regex=True)
          .str.replace(r"\.0$", "", regex=True))
    return c, pd.to_numeric(b, errors="coerce")

chunks = []
for ch in pd.read_csv(GWAS_PLOT_FILE, sep="\t", usecols=[v for v in COL.values() if v],
                      chunksize=2_000_000, low_memory=False):
    c, b = positions(ch)
    chunks.append(ch[(c == str(SNCA_CHR)) & b.between(lo, hi)].assign(CHR=c, BP=b))
reg = pd.concat(chunks, ignore_index=True)
assert len(reg), f"no variants in chr{SNCA_CHR}:{lo:,}-{hi:,} — check the build (hg38) or MarkerName format"

reg = reg.rename(columns={COL["A1"]: "Allele1", COL["A2"]: "Allele2", COL["beta"]: "Effect",
                          COL["se"]: "StdErr", COL["p"]: "P-value"})
if COL["freq"]:
    reg = reg.rename(columns={COL["freq"]: "Freq1"})
else:
    reg["Freq1"] = np.nan
for c in ("Allele1", "Allele2"):
    reg[c] = reg[c].astype(str).str.upper()                 # METAL writes alleles in lowercase
for c in ("Freq1", "Effect", "StdErr", "P-value"):
    reg[c] = pd.to_numeric(reg[c], errors="coerce")
reg = reg.dropna(subset=["P-value", "Effect", "StdErr"])
reg["BP"] = reg["BP"].astype(int)

if REQUIRE_BOTH_COHORTS and COL["dir"]:
    n0 = len(reg)
    reg = reg[~reg[COL["dir"]].astype(str).str.contains(r"\?")]
    print(f"Direction filter: {n0:,} → {len(reg):,} (present in both cohorts)")

reg = reg.sort_values("P-value").drop_duplicates("BP", keep="first")
print(f"variants in chr{SNCA_CHR}:{lo:,}-{hi:,}: {len(reg):,}")

# 3. r² to the lead from the chromosome-wide GP2 reference (NA if unavailable)
reg["R2"] = np.nan
plink1 = shutil.which("plink")
ld_bin, r2_flag = (plink1, "--r2") if plink1 else (PLINK, "--r2-unphased")
prefix = os.path.join(PLOT_DIR, "ld_lead")
r = run([ld_bin, "--bfile", SNCA_CHR_BFILE, r2_flag, "--ld-snp", target_id,
         "--ld-window-kb", PLOT_WINDOW_KB + 10, "--ld-window", 999999, "--ld-window-r2", 0,
         "--out", prefix], quiet=True, check=False)
ld_files = [f for f in glob.glob(prefix + "*") if re.search(r"\.(ld|vcor\d?)$", f)]
if r.returncode or not ld_files:
    print("LD step failed — plotting without r² colouring")
else:
    ld = pd.read_csv(ld_files[0], sep=r"\s+")
    ld.columns = [c.upper().lstrip("#") for c in ld.columns]
    bp_col = next(c for c in ld.columns if c in ("BP_B", "POS_B", "POSB"))
    r2_col = next(c for c in ld.columns if "R2" in c)
    ld = ld[[bp_col, r2_col]].rename(columns={bp_col: "BP", r2_col: "R2"})
    reg = reg.drop(columns="R2").merge(ld.drop_duplicates("BP"), on="BP", how="left")
    print(f"r² available for {reg['R2'].notna().sum():,}/{len(reg):,} variants")

# 4. Write the plot table
reg["LOG10P"]  = -np.log10(reg["P-value"].clip(lower=1e-300))
reg["rsID"]    = reg["BP"].map(HIGHLIGHT)
reg["is_lead"] = (reg["BP"] == LEAD_BP).astype(int)
reg.loc[reg["is_lead"] == 1, "R2"] = 1.0
reg[["CHR", "BP", "Allele1", "Allele2", "Freq1", "Effect", "StdErr", "P-value",
     "LOG10P", "R2", "rsID", "is_lead"]].to_csv(PLOT_TSV, sep="\t", index=False, na_rep="NA")
print("written:", PLOT_TSV)

print("\nhighlighted variants:")
for bp, rs in sorted(HIGHLIGHT.items()):
    print(f"  {rs:10s} chr{SNCA_CHR}:{bp:<12,} {'found' if bp in set(reg['BP']) else '*** NOT FOUND ***'}")

### 4b. Supplementary Figure 2 input (conditioned on rs2737030)

In [ ]:
PANELS_TSV   = os.path.join(PLOT_DIR, "SNCA_regional_2panel_input.tsv")
MATCH_PANELS = True                       # restrict panel a to COJO-matched variants so both panels match

cma_region, status = cojo_cond(SNCA_BFILE, SNCA_MA_AUG, [target_id],
                               os.path.join(OUT_DIR, f"SNCA_cond_region_on_{TARGET_RS}"))
assert status == "ok", status

a = pd.read_csv(PLOT_TSV, sep="\t", na_values="NA")
c = cma_region.rename(columns={"bp": "BP"}).merge(a[["BP", "R2", "rsID"]], on="BP", how="left")
c["LOG10P"]  = -np.log10(pd.to_numeric(c["pC"], errors="coerce").clip(lower=1e-300))
c["is_lead"] = 0                          # GCTA omits the conditioning SNP itself

if MATCH_PANELS:
    n0 = len(a)
    a = a[a["BP"].isin(set(c["BP"])) | (a["BP"] == LEAD_BP)]
    print(f"panel a restricted to COJO-matched variants: {n0:,} → {len(a):,}")

cols = ["BP", "LOG10P", "R2", "rsID", "is_lead"]
panels = pd.concat([a[cols].assign(panel="a"), c[cols].assign(panel="b")], ignore_index=True)
panels.to_csv(PANELS_TSV, sep="\t", index=False, na_rep="NA")
print("written:", PANELS_TSV)

print("\n-log10 P, unconditional → conditional:")
for bp, rs in sorted(HIGHLIGHT.items()):
    pa, pb = a.loc[a["BP"] == bp, "LOG10P"], c.loc[c["BP"] == bp, "LOG10P"]
    f = lambda s: f"{s.iloc[0]:5.2f}" if len(s) else "  n/a"
    print(f"  {rs:10s} {f(pa)} → {f(pb)}{'  (conditioned on)' if bp == LEAD_BP else ''}")